## 4. Konuşma Hafızası (Conversational Memory)
* Amaç: bir sohbetin bağlamını hatırlayan sistem oluşturması.
* Senaryo: Kullanıcı chatbot ile sohbet eder, önceki soruları hatırlayarak cevap verir.
* Kazandırılan: ConversationBufferMemory ve ChatPromptTemplate kullanımı.
* Kullanılan Bileşenler: ConversationChain, Memory

In [ ]:
# ============================================================
# 1) KURULUM
# ============================================================
!pip install -q "langchain>=0.2" "langchain-community>=0.2" \
               transformers accelerate sentencepiece

In [ ]:


# ============================================================
# 2) İÇE AKTARIMLAR
# ============================================================
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory, ConversationBufferWindowMemory, ConversationSummaryMemory

# ============================================================
# 3) LLM: HF ÜZERİNDEN AÇIK KAYNAK MODEL
# ============================================================
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"   # Alternatif: 

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

gen_pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.2,     # daha tutarlı cevaplar için düşük sıcaklık
    top_p=0.9,
    repetition_penalty=1.05,
)

llm = HuggingFacePipeline(pipeline=gen_pipe)

# ============================================================
# 4) HAFIZA SEÇENEKLERİ
# ============================================================
# 4a) Tam tampon (tüm geçmişi tutar) — eğitim için en görünür seçenek
buffer_memory = ConversationBufferMemory(
    memory_key="history",      # prompt'a eklenecek değişken adı
    input_key="input",         # kullanıcının mesajı
    return_messages=False      # True seçilirse ChatMessage objeleri tutulur
)

# 4b) (Opsiyonel) Sadece son N turu tutan hafıza
# window_memory = ConversationBufferWindowMemory(k=3, memory_key="history", input_key="input")

# 4c) (Opsiyonel) Özetleyerek hafıza — uzun diyaloğu kompakt özetler
# summary_memory = ConversationSummaryMemory(llm=llm, memory_key="history", input_key="input")

# ============================================================
# 5) KONUŞMA ZİNCİRİ
# ============================================================
# Varsayılan prompt: tarihçe + input'u birleştirip LLM'e verir.
# Özel bir prompt gerekmeden, hafıza otomatik enjekte edilir.
conv = ConversationChain(
    llm=llm,
    memory=buffer_memory,   # isterseniz window_memory veya summary_memory ile değiştirin
    verbose=True            # adımları (prompt+history) görmek için
)

# ============================================================
# 6) DENEYSEL DİYALOG (hafızayı test edelim)
# ============================================================
turns = [
    "Merhaba, ben finansal zaman serileri üzerinde çalışan bir araştırmacıyım.",
    "Bugün İstanbul'da hava nasıl bilmiyorum ama toplantım var. Bana hitap ederken kısaca 'hocam' diyebilirsin.",
    "Az önce bahsettiğim çalışma alanımı tek cümlede özetler misin?",
    "Bana nasıl hitap etmen gerektiğini hatırlıyor musun?",
    "Toplantıdan sonra hangi konuyu okumamı önermiştin? (Eğer önermediysen, şimdi öner.)"
]

for i, user_msg in enumerate(turns, 1):
    print(f"\n=== Kullanıcı ({i}) ===\n{user_msg}\n")
    reply = conv.predict(input=user_msg)
    print(f"--- Asistan ---\n{reply}")

# ============================================================
# 7) HAFIZAYI GÖRÜNTÜLEME (inceleme amaçlı)
# ============================================================
print("\n\n=== Hafıza (raw buffer) ===")
print(buffer_memory.buffer)

# Eğer window_memory veya summary_memory seçerseniz:
# print(window_memory.buffer) ya da print(summary_memory.buffer)
